# BERT Complete Pipeline: Preprocessing → Setup → Training

## Objective
End-to-end BERT model training pipeline for fake news classification combining:
1. **BERT Preprocessing**: Data loading, cleaning, and tokenization
2. **BERT Training Setup**: Class weight computation and loss function configuration
3. **BERT Model Training**: Training, validation, and comprehensive evaluation

**Key Features**:
- Lightweight text preprocessing preserving natural language
- Weighted loss handling class imbalance
- Proper validation integration during training
- Comprehensive evaluation with detailed metrics

---

# PART 1: PREPROCESSING

## Section 1: Import Libraries

In [ ]:
import pickle
import json
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import re

# PyTorch and HuggingFace
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Device: {device}")
print(f"  PyTorch version: {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")

## Section 2: Load Raw Datasets from TSV Files

In [ ]:
print("=" * 80)
print("LOADING RAW DATASETS FROM TSV FILES")
print("=" * 80)

# Define paths to data files
DATA_PATH = Path("data")
train_path = DATA_PATH / "train.tsv"
valid_path = DATA_PATH / "valid.tsv"
test_path = DATA_PATH / "test.tsv"

# Define column names based on the TSV structure
column_names = ["id", "label", "statement", "subjects", "speaker", "job", "state", "party", 
                "count_support", "count_refute", "count_discuss", "count_agree", "count_disagree", "context"]

# Load datasets
df_train = pd.read_csv(train_path, sep="\t", names=column_names, dtype={"statement": str, "label": str})
df_valid = pd.read_csv(valid_path, sep="\t", names=column_names, dtype={"statement": str, "label": str})
df_test = pd.read_csv(test_path, sep="\t", names=column_names, dtype={"statement": str, "label": str})

print("\n✓ Datasets loaded successfully\n")
print(f"Training set shape: {df_train.shape}")
print(f"Validation set shape: {df_valid.shape}")
print(f"Test set shape: {df_test.shape}")

# Display label distribution BEFORE consolidation
print("\n" + "=" * 80)
print("LABEL DISTRIBUTION (BEFORE CONSOLIDATION)")
print("=" * 80)
print("\nTraining Set:")
print(df_train["label"].value_counts())
print(f"\nValidation Set:")
print(df_valid["label"].value_counts())
print(f"\nTest Set:")
print(df_test["label"].value_counts())

## Section 3: Handle Missing Values

In [ ]:
print("=" * 80)
print("HANDLING MISSING VALUES")
print("=" * 80)

# Check for missing values
print("\nMissing values in Training set:")
missing_train = df_train.isnull().sum()
print(missing_train[missing_train > 0] if missing_train.sum() > 0 else "No missing values")

print("\nMissing values in Validation set:")
missing_valid = df_valid.isnull().sum()
print(missing_valid[missing_valid > 0] if missing_valid.sum() > 0 else "No missing values")

print("\nMissing values in Test set:")
missing_test = df_test.isnull().sum()
print(missing_test[missing_test > 0] if missing_test.sum() > 0 else "No missing values")

# Handle missing statement values
print("\n✓ Handling missing statement values (replace with empty string)...")
df_train["statement"] = df_train["statement"].fillna("")
df_valid["statement"] = df_valid["statement"].fillna("")
df_test["statement"] = df_test["statement"].fillna("")

print("  Train statements: None" if df_train["statement"].isnull().sum() == 0 else f"  Train: {df_train['statement'].isnull().sum()}")
print("  Valid statements: None" if df_valid["statement"].isnull().sum() == 0 else f"  Valid: {df_valid['statement'].isnull().sum()}")
print("  Test statements: None" if df_test["statement"].isnull().sum() == 0 else f"  Test: {df_test['statement'].isnull().sum()}")

print("\n✓ Missing values handled successfully")

## Section 4: Consolidate Labels (6 → 3 classes)

In [ ]:
print("=" * 80)
print("CONSOLIDATING LABELS (6 CLASSES → 3 CLASSES)")
print("=" * 80)

# Define label mapping
label_mapping = {
    'false': 'False',
    'pants-fire': 'False',
    'barely-true': 'Nuanced',
    'half-true': 'Nuanced',
    'mostly-true': 'True',
    'true': 'True'
}

# Apply mapping to all datasets
df_train['label'] = df_train['label'].map(label_mapping)
df_valid['label'] = df_valid['label'].map(label_mapping)
df_test['label'] = df_test['label'].map(label_mapping)

# Remove any unmapped labels (NaN)
initial_train = len(df_train)
initial_valid = len(df_valid)
initial_test = len(df_test)

df_train = df_train[df_train['label'].notna()]
df_valid = df_valid[df_valid['label'].notna()]
df_test = df_test[df_test['label'].notna()]

print(f"\nTraining set: {initial_train} → {len(df_train)} samples")
print(f"Validation set: {initial_valid} → {len(df_valid)} samples")
print(f"Test set: {initial_test} → {len(df_test)} samples")

print("\n✓ New label distribution (AFTER CONSOLIDATION):")
print("\nTraining Set:")
print(df_train['label'].value_counts())
print(f"\nValidation Set:")
print(df_valid['label'].value_counts())
print(f"\nTest Set:")
print(df_test['label'].value_counts())

## Section 5: Text Cleaning (Lightweight for BERT)

In [ ]:
print("=" * 80)
print("TEXT CLEANING (LIGHTWEIGHT FOR BERT)")
print("=" * 80)

def clean_text_for_bert(text):
    """
    Lightweight text cleaning for BERT.
    BERT wordpiece tokenizer handles most preprocessing.
    We only:
    - Convert to lowercase
    - Normalize whitespace
    """
    if not isinstance(text, str):
        return ""
    
    # Convert to lowercase
    text = text.lower()
    
    # Normalize whitespace (multiple spaces → single space)
    text = ' '.join(text.split())
    
    return text

# Apply cleaning to all datasets
print("\nCleaning statements...")
df_train['statement'] = df_train['statement'].apply(clean_text_for_bert)
df_valid['statement'] = df_valid['statement'].apply(clean_text_for_bert)
df_test['statement'] = df_test['statement'].apply(clean_text_for_bert)

print("  ✓ Train set: cleaned")
print("  ✓ Validation set: cleaned")
print("  ✓ Test set: cleaned")

# Show sample cleaned texts
print("\n✓ Sample cleaned texts:")
for i, text in enumerate(df_train['statement'].head(3), 1):
    print(f"\n{i}. {text[:100]}..." if len(text) > 100 else f"\n{i}. {text}")

## Section 6: Encode Labels

In [ ]:
print("=" * 80)
print("LABEL ENCODING")
print("=" * 80)

# Initialize label encoder
label_encoder = LabelEncoder()

# Fit on all unique labels
unique_labels = list(set(df_train['label'].tolist() + df_valid['label'].tolist() + df_test['label'].tolist()))
label_encoder.fit(unique_labels)

# Create label mappings
label_to_id = {label: idx for idx, label in enumerate(label_encoder.classes_)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

print("\n✓ Label mappings:")
for label, idx in label_to_id.items():
    print(f"  {label} → {idx}")

# Encode labels
df_train['label_id'] = df_train['label'].map(label_to_id)
df_valid['label_id'] = df_valid['label'].map(label_to_id)
df_test['label_id'] = df_test['label'].map(label_to_id)

print("\n✓ Labels encoded successfully")

## Section 7: BERT Tokenization

In [ ]:
print("=" * 80)
print("BERT TOKENIZATION")
print("=" * 80)

# Load tokenizer
print("\nLoading BERT tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print("✓ Tokenizer loaded")

# Tokenization parameters
MAX_LENGTH = 128

def tokenize_dataset(df, tokenizer, max_length=128):
    """
    Tokenize a dataset for BERT.
    """
    print(f"  Tokenizing {len(df)} samples...")
    
    encodings = tokenizer(
        df['statement'].tolist(),
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors=None
    )
    
    tokenized_data = {
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'labels': df['label_id'].tolist()
    }
    
    return tokenized_data

# Tokenize all datasets
print("\nTokenizing datasets...")
train_tokenized = tokenize_dataset(df_train, tokenizer, MAX_LENGTH)
valid_tokenized = tokenize_dataset(df_valid, tokenizer, MAX_LENGTH)
test_tokenized = tokenize_dataset(df_test, tokenizer, MAX_LENGTH)

print("\n✓ Tokenization complete")
print(f"\nTokenized data shapes:")
print(f"  Train: {np.array(train_tokenized['input_ids']).shape}")
print(f"  Valid: {np.array(valid_tokenized['input_ids']).shape}")
print(f"  Test: {np.array(test_tokenized['input_ids']).shape}")

## Section 8: Check for Data Leakage

In [ ]:
print("=" * 80)
print("DATA LEAKAGE VERIFICATION")
print("=" * 80)

# Convert tokenized data to arrays for comparison
train_texts = df_train['statement'].tolist()
valid_texts = df_valid['statement'].tolist()
test_texts = df_test['statement'].tolist()

# Check for overlaps
train_set = set(train_texts)
valid_set = set(valid_texts)
test_set = set(test_texts)

overlap_train_valid = len(train_set & valid_set)
overlap_train_test = len(train_set & test_set)
overlap_valid_test = len(valid_set & test_set)

print(f"\nOverlap between Train-Valid: {overlap_train_valid} samples")
print(f"Overlap between Train-Test: {overlap_train_test} samples")
print(f"Overlap between Valid-Test: {overlap_valid_test} samples")

if overlap_train_valid == 0 and overlap_train_test == 0 and overlap_valid_test == 0:
    print("\n✓ NO DATA LEAKAGE DETECTED - All datasets are independent")
else:
    print("\n⚠️  DATA LEAKAGE DETECTED - Check overlaps")

## Section 9: Save Preprocessed Data

In [ ]:
print("=" * 80)
print("SAVING PREPROCESSED DATA")
print("=" * 80)

# Create output directory
preprocessed_dir = Path("preprocessed_data")
preprocessed_dir.mkdir(exist_ok=True)

# Save tokenized data
print("\nSaving tokenized datasets...")
with open(preprocessed_dir / "train_tokenized.pkl", "wb") as f:
    pickle.dump(train_tokenized, f)
print("  ✓ train_tokenized.pkl")

with open(preprocessed_dir / "valid_tokenized.pkl", "wb") as f:
    pickle.dump(valid_tokenized, f)
print("  ✓ valid_tokenized.pkl")

with open(preprocessed_dir / "test_tokenized.pkl", "wb") as f:
    pickle.dump(test_tokenized, f)
print("  ✓ test_tokenized.pkl")

# Save label mappings
print("\nSaving label mappings...")
label_mappings = {
    "label_to_id": label_to_id,
    "id_to_label": id_to_label,
    "max_length": MAX_LENGTH
}

with open(preprocessed_dir / "label_mappings.json", "w") as f:
    json.dump(label_mappings, f, indent=2)
print("  ✓ label_mappings.json")

print("\n✓ All preprocessed data saved successfully")
print(f"\nOutput directory: {preprocessed_dir.absolute()}")

# PART 2: TRAINING SETUP

## Section 10: Load Preprocessed Data and Initialize Components

In [ ]:
print("=" * 80)
print("LOADING PREPROCESSED DATA AND MODEL COMPONENTS")
print("=" * 80)

# Load preprocessed data
preprocessed_dir = Path("preprocessed_data")

print("\nLoading tokenized datasets...")
with open(preprocessed_dir / "train_tokenized.pkl", "rb") as f:
    train_tokenized = pickle.load(f)
with open(preprocessed_dir / "valid_tokenized.pkl", "rb") as f:
    valid_tokenized = pickle.load(f)
with open(preprocessed_dir / "test_tokenized.pkl", "rb") as f:
    test_tokenized = pickle.load(f)

print(f"  ✓ Training: {train_tokenized['input_ids'].__len__()} samples")
print(f"  ✓ Validation: {len(valid_tokenized['input_ids'])} samples")
print(f"  ✓ Test: {len(test_tokenized['input_ids'])} samples")

# Load label mappings
with open(preprocessed_dir / "label_mappings.json", "r") as f:
    mappings = json.load(f)
    label_to_id = mappings["label_to_id"]
    id_to_label = {int(k): v for k, v in mappings["id_to_label"].items()}
    MAX_LENGTH = mappings["max_length"]

print(f"\n✓ Label mappings loaded")
print(f"  Labels: {list(label_to_id.keys())}")

# Define custom dataset class for dict-based format
class TokenizedDataset(Dataset):
    """Custom dataset class that returns dictionaries instead of tuples"""
    
    def __init__(self, tokenized_data):
        self.input_ids = tokenized_data['input_ids']
        self.attention_mask = tokenized_data['attention_mask']
        self.labels = tokenized_data['labels']
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.input_ids[idx], dtype=torch.long),
            'attention_mask': torch.tensor(self.attention_mask[idx], dtype=torch.long),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Create datasets
print("\nCreating datasets...")
train_dataset = TokenizedDataset(train_tokenized)
eval_dataset = TokenizedDataset(valid_tokenized)
test_dataset = TokenizedDataset(test_tokenized)

print(f"  ✓ Train dataset: {len(train_dataset)} samples")
print(f"  ✓ Eval dataset: {len(eval_dataset)} samples")
print(f"  ✓ Test dataset: {len(test_dataset)} samples")

# Load tokenizer
print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print(f"  ✓ Tokenizer loaded")

# Load model
print("\nLoading BERT model...")
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(id_to_label)
)
model.to(device)
print(f"  ✓ Model loaded: {type(model).__name__}")
print(f"  ✓ Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## Section 11: Compute Class Weights

In [ ]:
print("\n" + "=" * 80)
print("COMPUTING CLASS WEIGHTS")
print("=" * 80)

train_labels = np.array(train_tokenized['labels'])
class_weights_np = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights_tensor = torch.tensor(class_weights_np, dtype=torch.float32, device=device)

print("\n✓ Class weights (to handle imbalance):")
for label_id in range(len(id_to_label)):
    print(f"  {id_to_label[label_id]}: {class_weights_np[label_id]:.4f}")

# Test weighted loss with a sample batch
print("\n" + "=" * 80)
print("TESTING WEIGHTED LOSS FUNCTION")
print("=" * 80)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Create a test batch
test_logits = torch.randn(8, len(id_to_label), device=device)
test_labels = torch.tensor([0, 1, 2, 0, 1, 2, 0, 1], device=device)

test_loss = criterion(test_logits, test_labels)

print(f"\n✓ Weighted loss test:")
print(f"  Test batch loss: {test_loss.item():.6f}")
print(f"  Loss shape: {test_loss.shape}")
print(f"  Loss is scalar: {test_loss.dim() == 0}")

print("\n✓ Weighted loss function working correctly")

# PART 3: MODEL TRAINING

## Section 12: Define Custom Trainer with Weighted Loss

In [ ]:
print("=" * 80)
print("DEFINING CUSTOM TRAINER WITH WEIGHTED LOSS")
print("=" * 80)

class WeightedLossTrainer(Trainer):
    """
    Custom Trainer class that uses weighted CrossEntropyLoss
    to handle class imbalance in the training process.
    """
    
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.loss_fn = nn.CrossEntropyLoss(weight=self.class_weights)
        
        print(f"\n✓ WeightedLossTrainer initialized")
        print(f"  Class weights: {self.class_weights.cpu().numpy()}")
        print(f"  Loss function: CrossEntropyLoss (weighted)")
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = self.loss_fn(logits, labels)
        
        if loss.device != self.class_weights.device:
            self.class_weights = self.class_weights.to(loss.device)
        
        return (loss, outputs) if return_outputs else loss

print("\n✓ Custom Trainer class defined successfully")

## Section 13: Define Metrics Computation Function

In [ ]:
print("\n" + "=" * 80)
print("DEFINING METRICS COMPUTATION FUNCTION")
print("=" * 80)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    pred_labels = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, pred_labels)
    precision_macro = precision_score(labels, pred_labels, average='macro', zero_division=0)
    recall_macro = recall_score(labels, pred_labels, average='macro', zero_division=0)
    f1_macro = f1_score(labels, pred_labels, average='macro', zero_division=0)
    
    precision_weighted = precision_score(labels, pred_labels, average='weighted', zero_division=0)
    recall_weighted = recall_score(labels, pred_labels, average='weighted', zero_division=0)
    f1_weighted = f1_score(labels, pred_labels, average='weighted', zero_division=0)
    
    return {
        'accuracy': accuracy,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        'precision_weighted': precision_weighted,
        'recall_weighted': recall_weighted,
        'f1_weighted': f1_weighted,
    }

print("\n✓ Metrics computation function defined")
print(f"\nMetrics computed:")
print(f"  - Accuracy")
print(f"  - Precision (macro & weighted)")
print(f"  - Recall (macro & weighted)")
print(f"  - F1-score (macro & weighted)")

## Section 14: Configure Training Arguments

In [ ]:
print("\n" + "=" * 80)
print("CONFIGURING TRAINING ARGUMENTS")
print("=" * 80)

output_dir = "./bert_model_checkpoints"

training_args = TrainingArguments(
    output_dir=output_dir,
    logging_dir='./logs',
    logging_strategy='epoch',
    log_level='info',
    
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,
    
    eval_strategy='epoch',
    save_strategy='epoch',
    
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    save_total_limit=2,
    
    seed=42,
    fp16=torch.cuda.is_available(),
    dataloader_pin_memory=True,
    remove_unused_columns=False,
)

print(f"\n✓ Training arguments configured")
print(f"\nTraining Configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Train batch size: {training_args.per_device_train_batch_size}")
print(f"  Eval batch size: {training_args.per_device_eval_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Eval strategy: {training_args.eval_strategy}")
print(f"  Save strategy: {training_args.save_strategy}")
print(f"  Load best model: {training_args.load_best_model_at_end}")
print(f"  Best model metric: {training_args.metric_for_best_model}")
print(f"  Output directory: {output_dir}")

## Section 15: Initialize the Trainer

In [ ]:
print("\n" + "=" * 80)
print("INITIALIZING TRAINER")
print("=" * 80)

data_collator = DataCollatorWithPadding(tokenizer)

trainer = WeightedLossTrainer(
    class_weights=class_weights_tensor,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

print(f"\n✓ Trainer initialized successfully")
print(f"\nTrainer Configuration:")
print(f"  Model: {type(trainer.model).__name__}")
print(f"  Training dataset: {len(trainer.train_dataset)} samples")
print(f"  Eval dataset: {len(trainer.eval_dataset)} samples")
print(f"  Compute metrics: {trainer.compute_metrics is not None}")
print(f"  Device: {trainer.model.device}")
print(f"  Mixed precision: {training_args.fp16}")

## Section 16: Train the Model

In [ ]:
print("\n" + "=" * 80)
print("TRAINING THE MODEL")
print("=" * 80)

print(f"\nStarting training with validation tracking...")
print(f"  Total epochs: {training_args.num_train_epochs}")
print(f"  Evaluation at: Every epoch")
print(f"  Best model selected by: {training_args.metric_for_best_model}")

train_result = trainer.train()

print(f"\n" + "=" * 80)
print("✓ TRAINING COMPLETED SUCCESSFULLY")
print("=" * 80)

print(f"\\nTraining Results:")
print(f"  Final training loss: {train_result.training_loss:.6f}")
if hasattr(train_result, 'best_metric'):
    print(f"  Best {training_args.metric_for_best_model}: {train_result.best_metric:.6f}")
if hasattr(train_result, 'best_epoch'):
    print(f"  Best epoch: {int(train_result.best_epoch)}")
if hasattr(train_result, 'global_step'):
    print(f"  Total training steps: {train_result.global_step}")

print(f"\n✓ Best model automatically loaded from checkpoint")

## Section 17: Evaluate on Validation Set

In [ ]:
print("\n" + "=" * 80)
print("EVALUATING ON VALIDATION SET")
print("=" * 80)

print(f"\nEvaluating on validation set ({len(eval_dataset)} samples)...")
eval_results = trainer.evaluate(eval_dataset)

print(f"\n✓ Validation Evaluation Results:")
print(f"\n  Loss: {eval_results.get('eval_loss', 'N/A'):.6f}")
print(f"\nAccuracy Metrics:")
print(f"  Accuracy: {eval_results.get('eval_accuracy', 0):.4f}")

print(f"\nPrecision (Macro): {eval_results.get('eval_precision_macro', 0):.4f}")
print(f"Recall (Macro): {eval_results.get('eval_recall_macro', 0):.4f}")
print(f"F1-score (Macro): {eval_results.get('eval_f1_macro', 0):.4f}")

print(f"\nPrecision (Weighted): {eval_results.get('eval_precision_weighted', 0):.4f}")
print(f"Recall (Weighted): {eval_results.get('eval_recall_weighted', 0):.4f}")
print(f"F1-score (Weighted): {eval_results.get('eval_f1_weighted', 0):.4f}")

val_accuracy = eval_results.get('eval_accuracy', 0)
val_f1 = eval_results.get('eval_f1_macro', 0)

## Section 18: Evaluate on Test Set

In [ ]:
print("\n" + "=" * 80)
print("EVALUATING ON TEST SET")
print("=" * 80)

print(f"\nGenerating predictions on test set ({len(test_dataset)} samples)...")
predictions = trainer.predict(test_dataset)

test_preds = np.argmax(predictions.predictions, axis=1)
test_labels = predictions.label_ids

print(f"✓ Predictions generated for {len(test_preds)} test samples")

print(f"\nComputing test metrics...")
test_accuracy = np.mean(test_preds == test_labels)

test_precision_macro = precision_score(test_labels, test_preds, average='macro', zero_division=0)
test_recall_macro = recall_score(test_labels, test_preds, average='macro', zero_division=0)
test_f1_macro = f1_score(test_labels, test_preds, average='macro', zero_division=0)

test_precision_weighted = precision_score(test_labels, test_preds, average='weighted', zero_division=0)
test_recall_weighted = recall_score(test_labels, test_preds, average='weighted', zero_division=0)
test_f1_weighted = f1_score(test_labels, test_preds, average='weighted', zero_division=0)

print(f"\n✓ Test Set Evaluation Results:")
print(f"\n  Accuracy: {test_accuracy:.4f}")

print(f"\nPrecision (Macro): {test_precision_macro:.4f}")
print(f"Recall (Macro): {test_recall_macro:.4f}")
print(f"F1-score (Macro): {test_f1_macro:.4f}")

print(f"\nPrecision (Weighted): {test_precision_weighted:.4f}")
print(f"Recall (Weighted): {test_recall_weighted:.4f}")
print(f"F1-score (Weighted): {test_f1_weighted:.4f}")

# Confusion matrix
print(f"\n" + "=" * 80)
print("CONFUSION MATRIX (Test Set)")
print("=" * 80)
conf_matrix = confusion_matrix(test_labels, test_preds)
label_names = [id_to_label[i] for i in range(len(id_to_label))]

print(f"\nConfusion Matrix:")
print(f"{'':15} {label_names[0]:>12} {label_names[1]:>12} {label_names[2]:>12}")
for i, label in enumerate(label_names):
    print(f"{label:15} {conf_matrix[i, 0]:>12} {conf_matrix[i, 1]:>12} {conf_matrix[i, 2]:>12}")

# Per-class metrics
print(f"\n" + "=" * 80)
print("DETAILED CLASSIFICATION REPORT (Test Set)")
print("=" * 80)
print(classification_report(test_labels, test_preds, target_names=label_names, digits=4))

test_results = {
    'accuracy': test_accuracy,
    'precision_macro': test_precision_macro,
    'recall_macro': test_recall_macro,
    'f1_macro': test_f1_macro,
    'precision_weighted': test_precision_weighted,
    'recall_weighted': test_recall_weighted,
    'f1_weighted': test_f1_weighted
}

## Section 19: Results Summary and Comparison

In [ ]:
print("\n" + "=" * 80)
print("COMPREHENSIVE RESULTS SUMMARY")
print("=" * 80)

training_history = trainer.state.log_history
best_epoch = trainer.state.best_model_checkpoint.split('-')[-1] if trainer.state.best_model_checkpoint else 'Final'

print(f"\nTraining Information:")
print(f"  Number of epochs: {training_args.num_train_epochs}")
print(f"  Best model checkpoint: Epoch {best_epoch}")
print(f"  Best F1-macro on validation: {train_result.best_metric:.6f}")
print(f"  Final training loss: {train_result.training_loss:.6f}")

# Comparison table
print(f"\n" + "=" * 80)
print("PERFORMANCE COMPARISON ACROSS SPLITS")
print("=" * 80)

comparison_data = {
    'Metric': ['Accuracy', 'Precision (Macro)', 'Recall (Macro)', 'F1 (Macro)',
               'Precision (Weighted)', 'Recall (Weighted)', 'F1 (Weighted)'],
    'Validation': [
        eval_results.get('eval_accuracy', 0),
        eval_results.get('eval_precision_macro', 0),
        eval_results.get('eval_recall_macro', 0),
        eval_results.get('eval_f1_macro', 0),
        eval_results.get('eval_precision_weighted', 0),
        eval_results.get('eval_recall_weighted', 0),
        eval_results.get('eval_f1_weighted', 0)
    ],
    'Test': [
        test_results['accuracy'],
        test_results['precision_macro'],
        test_results['recall_macro'],
        test_results['f1_macro'],
        test_results['precision_weighted'],
        test_results['recall_weighted'],
        test_results['f1_weighted']
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n" + comparison_df.to_string(index=False))

# Gap analysis
print(f"\n" + "=" * 80)
print("GAP ANALYSIS (Test - Validation)")
print("=" * 80)
print(f"\nAccuracy gap: {test_results['accuracy'] - eval_results.get('eval_accuracy', 0):+.4f}")
print(f"F1 (Macro) gap: {test_results['f1_macro'] - eval_results.get('eval_f1_macro', 0):+.4f}")
print(f"Precision (Weighted) gap: {test_results['precision_weighted'] - eval_results.get('eval_precision_weighted', 0):+.4f}")

# Generalization analysis
val_f1 = eval_results.get('eval_f1_macro', 0)
test_f1 = test_results['f1_macro']
f1_gap = val_f1 - test_f1

print(f"\n" + "=" * 80)
print("GENERALIZATION ANALYSIS")
print("=" * 80)

if f1_gap < -0.05:
    print(f"\n⚠️  Overfitting Detected: Validation F1 > Test F1 by {abs(f1_gap):.4f}")
    print("   Model may not generalize well to new data")
elif f1_gap > 0.05:
    print(f"\n✓ Good Generalization: Test F1 > Validation F1 by {abs(f1_gap):.4f}")
    print("   Model generalizes well to unseen data")
else:
    print(f"\n✓ Balanced Performance: F1 gap of {abs(f1_gap):.4f} indicates good generalization")

# Model information
print(f"\n" + "=" * 80)
print("MODEL INFORMATION")
print("=" * 80)
print(f"\nModel: {training_args.output_dir}")
print(f"Number of parameters: {model.num_parameters():,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Batch size (train): {training_args.per_device_train_batch_size}")
print(f"Batch size (eval): {training_args.per_device_eval_batch_size}")

# Final recommendations
print(f"\n" + "=" * 80)
print("FINAL RECOMMENDATIONS")
print("=" * 80)

if test_results['accuracy'] > 0.85:
    print(f"\n✓ Excellent Performance: Test accuracy of {test_results['accuracy']:.4f}")
    print("  Model is ready for production use")
elif test_results['accuracy'] > 0.75:
    print(f"\n✓ Good Performance: Test accuracy of {test_results['accuracy']:.4f}")
    print("  Model can be deployed with monitoring")
else:
    print(f"\n⚠️  Moderate Performance: Test accuracy of {test_results['accuracy']:.4f}")
    print("  Consider improving data or model architecture")

if max(test_results['precision_macro'], test_results['recall_macro']) - min(test_results['precision_macro'], test_results['recall_macro']) > 0.1:
    print(f"\n⚠️  Precision-Recall Imbalance Detected")
    print(f"   Precision: {test_results['precision_macro']:.4f}, Recall: {test_results['recall_macro']:.4f}")
    print("  Consider adjusting decision threshold or class weights")

print(f"\n✓ Complete BERT training pipeline executed successfully!")
print(f"  Final test F1-score (macro): {test_results['f1_macro']:.4f}")
print(f"  Model saved to: {training_args.output_dir}")